# Week 11 Representation Theory Worksheet

## Real Elements in $S_5$, $D_{10}$, and $\mathbb{Z}/5\mathbb{Z}$

An element $g \in G$ is called **real** if its inverse $g^{-1}$ is in its conjugacy class. That is, there exists $h \in G$ such that $g^{-1} = hgh^{-1}$. Alternatively, this means the conjugacy class of $g$ and $g^{-1}$ are the same.

In [2]:
def find_real_classes(G, name="Group"):
    print(f"--- Real Classes in {name} --- ")
    real_classes = []
    non_real_classes = []
    
    for c in G.conjugacy_classes():
        g = c.representative()
        g_inv = g**(-1)
        # Check if the inverse is in the same conjugacy class
        if g_inv in c:
            real_classes.append(c)
            print(f"Real: class of size {c.cardinality()} with representative {g}")
        else:
            non_real_classes.append(c)
            print(f"NOT Real: class of size {c.cardinality()} with representative {g}")
            
    print(f"\nSummary for {name}:")
    print(f"Total classes: {len(G.conjugacy_classes())}. Real: {len(real_classes)}. Not real: {len(non_real_classes)}.\n")

### 1. The Symmetric Group $S_5$
In any symmetric group $S_n$, two elements are conjugate if and only if they have the same cycle type. Since an element and its inverse always have the same cycle type, **every element in $S_n$ is real**.

In [3]:
S5 = SymmetricGroup(5)
find_real_classes(S5, "S5")

--- Real Classes in S5 --- 
Real: class of size 1 with representative ()
Real: class of size 10 with representative (1,2)
Real: class of size 15 with representative (1,2)(3,4)
Real: class of size 20 with representative (1,2,3)
Real: class of size 20 with representative (1,2,3)(4,5)
Real: class of size 30 with representative (1,2,3,4)
Real: class of size 24 with representative (1,2,3,4,5)

Summary for S5:
Total classes: 7. Real: 7. Not real: 0.



### 2. The Dihedral Group $D_{10}$ (Symmetries of a Pentagon)
In the dihedral group $D_{2n}$, reflections are their own inverse (hence trivially real). For rotations, $r^{-1} = r^{n-1}$, and since $s \cdot r \cdot s^{-1} = r^{-1}$ (where $s$ is a reflection), all rotations are conjugate to their inverses. Thus, **every element in $D_{2n}$ is real**.

In [4]:
# In Sage, DihedralGroup(n) creates the group of order 2n.
D10 = DihedralGroup(5)
find_real_classes(D10, "D10 (order 10)")

--- Real Classes in D10 (order 10) --- 
Real: class of size 1 with representative ()
Real: class of size 5 with representative (2,5)(3,4)
Real: class of size 2 with representative (1,2,3,4,5)
Real: class of size 2 with representative (1,3,5,2,4)

Summary for D10 (order 10):
Total classes: 4. Real: 4. Not real: 0.



### 3. The Cyclic Group $\mathbb{Z}/5\mathbb{Z}$
In an abelian group, the conjugacy class of an element $g$ is just $\{g\}$. Therefore, $g$ is real if and only if $g = g^{-1}$, which means $g^2 = e$ (so $g$ is the identity or has order 2). In $\mathbb{Z}/5\mathbb{Z}$, there are no elements of order 2, so the **only real element is the identity**.

In [5]:
Z5 = CyclicPermutationGroup(5)
find_real_classes(Z5, "Z/5Z")

--- Real Classes in Z/5Z --- 
Real: class of size 1 with representative ()
NOT Real: class of size 1 with representative (1,2,3,4,5)
NOT Real: class of size 1 with representative (1,3,5,2,4)
NOT Real: class of size 1 with representative (1,4,2,5,3)
NOT Real: class of size 1 with representative (1,5,4,3,2)

Summary for Z/5Z:
Total classes: 5. Real: 1. Not real: 4.



## Theorem: Number of Real Characters = Number of Real Conjugacy Classes

Let $X$ be the character table of a group $G$ with entries $\chi_i(g_j)$, and let $\overline{X}$ be its complex conjugate. We investigate this theorem computationally through matrices.

### 1. Row Perspective (Irreducible Characters)
If $\chi_i$ is a real character, $\chi_i = \overline{\chi_i}$. If it is not real, $\overline{\chi_i}$ is a different irreducible character. Thus, complex conjugating the character table permutes its rows. 
This means we have a permutation matrix $P$ such that $\overline{X} = PX$.
The trace of $P$ counts the invariant rows, which corresponds to the number of **real irreducible characters**.

### 2. Column Perspective (Conjugacy Classes)
For a conjugacy class $g_j$, its entries across all characters are $\chi_i(g_j)$. We know by character properties that $\overline{\chi_i(g_j)} = \chi_i(g_j^{-1})$. 
If $g_j$ is a real conjugacy class, $g_j$ is conjugate to $g_j^{-1}$, so $\overline{\chi_i(g_j)} = \chi_i(g_j)$ (invariant column). If it is not real, $g_j^{-1}$ represents a different class. Thus, complex conjugating the table permutes its columns. 
This means we have a permutation matrix $Q$ such that $\overline{X} = XQ$.
The trace of $Q$ counts the invariant columns, which corresponds to the number of **real conjugacy classes**.

### Conclusion
Since $\overline{X} = PX = XQ$, we can multiply by $X^{-1}$ on the left to get $Q = X^{-1}PX$. This means $P$ and $Q$ are similar matrices, hence they have the **same trace**. This beautifully proves that the number of real characters equals the number of real conjugacy classes!

In [6]:
def analyze_real_theorem(G, name="Group"):
    print(f"--- Analyzing Theorem for {name} ---")
    
    # Get Character Table as a Sage matrix
    X_list = G.character_table()
    X = matrix(X_list)
    print("Character Table X:")
    print(X)
    
    # Form the complex conjugate character table 
    # We iterate and apply .conjugate() because entries might be cyclotomic integers
    X_bar = matrix([[val.conjugate() for val in row] for row in X_list])
    
    print("\nConjugate Character Table X_bar:")
    print(X_bar)
    
    # --- Row Perspective ---
    # X_bar = P * X  =>  P = X_bar * X^-1
    # This permutation matrix swaps non-real characters
    P = X_bar * X.inverse()
    print("\nRow Permutation Matrix P (X_bar = P * X):")
    print(P)
    trace_P = P.trace()
    print(f"Trace of P (Number of Real Characters) = {trace_P}")
    
    # --- Column Perspective ---
    # X_bar = X * Q  =>  Q = X^-1 * X_bar
    # This permutation matrix swaps non-real conjugacy classes
    Q = X.inverse() * X_bar
    print("\nColumn Permutation Matrix Q (X_bar = X * Q):")
    print(Q)
    trace_Q = Q.trace()
    print(f"Trace of Q (Number of Real Conj Classes) = {trace_Q}")
    
    print(f"\nTheorem Verification: trace(P) == trace(Q) evaluates to {trace_P == trace_Q}\n")

### Example 1: The Symmetric Group $S_3$
For $S_3$, every element is conjugate to its inverse, so every class is real. Thus, we expect all characters to be real as well, meaning $P$ and $Q$ should just be the Identity matrix.

In [7]:
S3 = SymmetricGroup(3)
analyze_real_theorem(S3, "S3")

--- Analyzing Theorem for S3 ---
Character Table X:
[ 1 -1  1]
[ 2  0 -1]
[ 1  1  1]

Conjugate Character Table X_bar:
[ 1 -1  1]
[ 2  0 -1]
[ 1  1  1]

Row Permutation Matrix P (X_bar = P * X):
[1 0 0]
[0 1 0]
[0 0 1]
Trace of P (Number of Real Characters) = 3

Column Permutation Matrix Q (X_bar = X * Q):
[1 0 0]
[0 1 0]
[0 0 1]
Trace of Q (Number of Real Conj Classes) = 3

Theorem Verification: trace(P) == trace(Q) evaluates to True



### Example 2: The Cyclic Group $\mathbb{Z}/6\mathbb{Z}$
In $\mathbb{Z}/6\mathbb{Z}$, the elements are $0, 1, 2, 3, 4, 5$. The only real classes (where $g = g^{-1}$) are $0$ and $3$. Thus, the number of real conjugacy classes is 2. We expect $P$ and $Q$ to be non-trivial permutation matrices that swap non-real classes/characters, while leaving exactly 2 invariant (yielding a trace of 2).

In [8]:
Z6 = CyclicPermutationGroup(6)
analyze_real_theorem(Z6, "Z/6Z")

--- Analyzing Theorem for Z/6Z ---
Character Table X:
[         1          1          1          1          1          1]
[         1         -1          1         -1          1         -1]
[         1 -zeta3 - 1      zeta3          1 -zeta3 - 1      zeta3]
[         1  zeta3 + 1      zeta3         -1 -zeta3 - 1     -zeta3]
[         1      zeta3 -zeta3 - 1          1      zeta3 -zeta3 - 1]
[         1     -zeta3 -zeta3 - 1         -1      zeta3  zeta3 + 1]

Conjugate Character Table X_bar:
[         1          1          1          1          1          1]
[         1         -1          1         -1          1         -1]
[         1      zeta3 -zeta3 - 1          1      zeta3 -zeta3 - 1]
[         1     -zeta3 -zeta3 - 1         -1      zeta3  zeta3 + 1]
[         1 -zeta3 - 1      zeta3          1 -zeta3 - 1      zeta3]
[         1  zeta3 + 1      zeta3         -1 -zeta3 - 1     -zeta3]

Row Permutation Matrix P (X_bar = P * X):
[1 0 0 0 0 0]
[0 1 0 0 0 0]
[0 0 0 0 1 0]
[0 0 0 0 0 

For $(\mathbb{Z}/2\mathbb{Z}) \times (\mathbb{Z}/2\mathbb{Z})$, every element is its own inverse ($g^2 = e$). Therefore, all 4 conjugacy classes are real, and you will see the trace of both $P$ and $Q$ is 4.

In [9]:
V4 = groups.permutation.KleinFour()
analyze_real_theorem(V4, "(Z/2Z)x(Z/2Z)")

--- Analyzing Theorem for (Z/2Z)x(Z/2Z) ---
Character Table X:
[ 1  1  1  1]
[ 1 -1 -1  1]
[ 1 -1  1 -1]
[ 1  1 -1 -1]

Conjugate Character Table X_bar:
[ 1  1  1  1]
[ 1 -1 -1  1]
[ 1 -1  1 -1]
[ 1  1 -1 -1]

Row Permutation Matrix P (X_bar = P * X):
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]
Trace of P (Number of Real Characters) = 4

Column Permutation Matrix Q (X_bar = X * Q):
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]
Trace of Q (Number of Real Conj Classes) = 4

Theorem Verification: trace(P) == trace(Q) evaluates to True



In a direct product $G \times H$, an element $(g, h)$ is real if and only if $g$ is real in $G$ and $h$ is real in $H$.

For cyclic groups $\mathbb{Z}/k\mathbb{Z}$, an element $x$ is real only if $2x \equiv 0 \pmod k$:

If $k$ is odd, there is only 1 real element (the identity).
If $k$ is even, there are exactly 2 real elements (the identity and $k/2$).

In [10]:
def analyze_cyclic_product(m, n):
    # Create the direct product of two cyclic groups
    C_m = CyclicPermutationGroup(m)
    C_n = CyclicPermutationGroup(n)
    G = direct_product_permgroups([C_m, C_n])
    
    # Run the theorem analysis
    analyze_real_theorem(G, f"Z/{m}Z x Z/{n}Z")


Examples:
One odd, one even: expect trace to be 1 * 2 = 2

In [11]:
analyze_cyclic_product(3, 4)

--- Analyzing Theorem for Z/3Z x Z/4Z ---
Character Table X:
[                 1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1]
[                 1                 -1                  1                 -1                  1                 -1                  1                 -1                  1                 -1                  1                 -1]
[                 1                  1                  1                  1       zeta12^2 - 1       zeta12^2 - 1       zeta12^2 - 1       zeta12^2 - 1          -zeta12^2          -zeta12^2          -zeta12^2          -zeta12^2]
[                 1                 -1                  1                 -1       zeta12^2 - 1      -zeta12^2 + 1       zeta12^2 - 1      -zeta12^2 + 1          -zeta12^2           zeta12^2          -zeta12^2           zeta12^2]
[                 1

In [12]:
# Both odd: expect trace to be 1 * 1 = 1
analyze_cyclic_product(3, 5)


--- Analyzing Theorem for Z/3Z x Z/5Z ---
Character Table X:
[                                                        1                                                         1                                                         1                                                         1                                                         1                                                         1                                                         1                                                         1                                                         1                                                         1                                                         1                                                         1                                                         1                                                         1                                                         1]
[                                                        1         

In [13]:
# Both even: expect trace to be 2 * 2 = 4
analyze_cyclic_product(4, 6)

--- Analyzing Theorem for Z/4Z x Z/6Z ---
Character Table X:
[                 1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1                  1]
[                 1                 -1                  1                 -1                  1                 -1                  1                 -1                  1                 -1                  1                 -1                  1                 -1                  1                 -1                  1                 -1                  1                 -1                  1                 -1                  1                 -1]
[                 1    

Lets now do another proof-by-example as we just did. The proposition now is 

For V a CG module with chracter \chi, 
1. The \mathbb R G module V_\bR has chracter \chi+\overline{\chi} and \dim V=\dim V_\bR.
2. READ OTEHR PART



## Exercises - 2026-04-08

In this section, we explore the properties of Dihedral groups $D_{2n}$, with a focus on involution counts and Frobenius-Schur indicators. We verify the following for $D_4, D_6, D_8$ and $D_{10}$:

1. **The number of involution elements**: $n_{1} = \#\{g \in G : g^2 = 1\}$.
2. **The number of conjugacy classes** of the group.
3. **The dimensions** of the irreducible representations.
4. **The Frobenius-Schur indicators** $\nu(\chi)$ for each irreducible character.
5. **Sum Formula**: Verify that $\sum_{\chi \in \text{Irr}(G)} \nu(\chi) \chi(1) = n_1$.

In [8]:
def verify_dihedral_properties(n, name):
    G = DihedralGroup(n)
    print(f"--- Analysis for {name} (Order {G.order()}) ---")
    
    # 1. Number of g such that g^2 = 1
    involutions = [g for g in G if g**2 == G.one()]
    n1 = len(involutions)
    print(f"1. Number of elements satisfying g^2 = 1: {n1}")
    if n % 2 != 0:
        print(f"   Check (n odd): Is n+1 ({n+1})? {n1 == n+1}")
    else:
        print(f"   Note (n even): Count is {n1} (expected n+2 = {n+2})")
    print(f"   The elements are: {involutions}")
    
    # 2. Number of conjugacy classes
    n_classes = len(G.conjugacy_classes())
    print(f"2. Number of conjugacy classes: {n_classes}")
    if n % 2 != 0:
        expected = (n-1)/2 + 2
        print(f"   Check (n odd): Is (n-1)/2 + 2 ({expected})? {n_classes == expected}")
    
    # 3. Dimensions of irreps
    chars = G.irreducible_characters()
    dims = [chi(G.one()) for chi in chars]
    print(f"3. Dimensions of irreps: {dims}")
    
    # 4. Frobenius-Schur indicators (computed manually)
    def calculate_fs_indicator(chi, group):
        order = group.order()
        s = sum(chi(g**2) for g in group)
        return s / order

    indicators = [calculate_fs_indicator(chi, G) for chi in chars]
    print(f"4. Frobenius-Schur indicators: {indicators}")
    
    # 5. Sum of (dim * indicator)
    the_sum = sum(dims[i] * indicators[i] for i in range(len(chars)))
    print(f"5. Sum of (dim * indicator): {the_sum}")
    print(f"   Formula holds (Sum == n1): {the_sum == n1}")
    print("\n")

In [9]:
# n=2 corresponds to D4
verify_dihedral_properties(2, "D4")

# n=3 corresponds to D6
verify_dihedral_properties(3, "D6")

# n=4 corresponds to D8
verify_dihedral_properties(4, "D8")

# n=5 corresponds to D10
verify_dihedral_properties(5, "D10")

--- Analysis for D4 (Order 4) ---
1. Number of elements satisfying g^2 = 1: 4
   Note (n even): Count is 4 (expected n+2 = 4)
   The elements are: [(), (3,4), (1,2), (1,2)(3,4)]
2. Number of conjugacy classes: 4
3. Dimensions of irreps: [1, 1, 1, 1]
4. Frobenius-Schur indicators: [1, 1, 1, 1]
5. Sum of (dim * indicator): 4
   Formula holds (Sum == n1): True


--- Analysis for D6 (Order 6) ---
1. Number of elements satisfying g^2 = 1: 4
   Check (n odd): Is n+1 (4)? True
   The elements are: [(), (2,3), (1,3), (1,2)]
2. Number of conjugacy classes: 3
   Check (n odd): Is (n-1)/2 + 2 (3)? True
3. Dimensions of irreps: [1, 2, 1]
4. Frobenius-Schur indicators: [1, 1, 1]
5. Sum of (dim * indicator): 4
   Formula holds (Sum == n1): True


--- Analysis for D8 (Order 8) ---
1. Number of elements satisfying g^2 = 1: 6
   Note (n even): Count is 6 (expected n+2 = 6)
   The elements are: [(), (1,3)(2,4), (2,4), (1,3), (1,4)(2,3), (1,2)(3,4)]
2. Number of conjugacy classes: 5
3. Dimensions of irre